# 05 · Estudio de ablación — documentación y reporte

**Pregunta que responde.** El Prof. Marcos Alania (plan de mejora 5.4) pidió aislar el aporte de cada componente: *1) modelo base → 2) + codificación cíclica → 3) + Weighted Focal Loss*. Se extendió con precursores (TEC, ROTEC, ROTI, índices globales) para responder además: *¿la red aprovecha ROTI o es una asunción?*

**Qué hace este notebook.** Documenta los pasos seguidos, muestra los scripts en el orden en que se ejecutaron y **genera las tablas y figuras a partir del registro de experimentos, sin volver a entrenar**.

> ⚠️ **No reentrenar desde aquí.** Los entrenamientos ya se ejecutaron con los scripts de la sección 2. Repetirlos agregaría filas duplicadas al registro y alteraría las medias y desviaciones entre semillas.

| Elemento | Valor |
|---|---|
| Datos | S4 reconstruido de Jicamarca + precursores causalizados, periodo común 26-ene-2025 a 21-jul-2026 |
| Partición | Cronológica Out-of-Time, corte por día ionosférico (12 UT = 07 LT): Train hasta 09-dic-2025, Validación hasta 01-mar-2026 |
| Modelo | LSTM apilada 64/64, dropout 0.3, Dense 64 → 20 |
| Horizonte | 20 minutos |
| Evaluación | Validación (Test se reserva para el modelo final, notebook 09) |
| Semillas | 42, 123, 7 |
| Configuración | `config/ablacion_2025_2026.yaml` |

## 1. Pasos seguidos (en orden)

| # | Paso | Herramienta | Resultado |
|---|---|---|---|
| 1 | Construir el dataset común S4 + precursores, causalizar índices y TEC/ROTI, particionar y auditar | `notebooks/02_split_y_eda.ipynb` | Fechas de corte copiadas al YAML |
| 2 | Corregir la unión de datos: el CSV multivariable no tiene datos de 23:46 a 00:00 UT y el *inner join* cortaba cada noche | `src/s4/features.py` (unión por la izquierda + último valor ≤ 60 min) | Cobertura de minutos de evento: de ≈ 0 % a 100 % |
| 3 | Elegir la ventana de entrada (LOOKBACK) con S4 + hora + WFL | `scripts/correr_lookback.py` | LOOKBACK = **60 min** (fijado en el YAML) |
| 4 | Ablación principal: baselines + 6 variantes × 3 semillas | `scripts/correr_ablacion.py` | `reports/tables/tabla_ablacion.csv` |
| 5 | Reparar el registro de experimentos (filas con distinto número de columnas) | `scripts/reparar_registro.py` + `src/s4/registry.py` y `evaluate.py` corregidos | Registro alineado; tablas regeneradas |
| 6 | Ablación complementaria: S4 + WFL sin hora y precursores con ventana de 180 min | `scripts/correr_ablacion_extra.py` | `reports/tables/tabla_ablacion_extra.csv` |
| 7 | Reporte: tablas y figuras para la tesis | **este notebook** | `fig_ablacion_hss.png`, `fig_ablacion_precision_pod.png` |

## 2. Scripts empleados (en el orden de ejecución)

Todos se ejecutaron **desde la raíz del proyecto** (`~/Documents/scintia-s4`), en segundo plano para que no dependieran de VS Code:

```bash
conda activate s4_env
nohup python scripts/<script>.py > experiments/log_<nombre>.txt 2>&1 &
tail -f experiments/log_<nombre>.txt      # seguir el avance
```

### 2.1 `scripts/correr_lookback.py` — sensibilidad de la ventana (paso 3)

Compara LOOKBACK ∈ {30, 60, 120, 180, 240, 360} min sobre **los mismos instantes objetivo** (`inicios_comunes`), con S4 + hora + WFL y 3 semillas.

```python
import sys, warnings
sys.path.insert(0, "src"); warnings.filterwarnings("ignore")
from s4.config import cargar_config, variante, ruta
from s4.pipeline import preparar_datos, correr_variante
from s4.windows import inicios_comunes
from s4.registry import leer, resumir_semillas
from s4.train import configurar_gpu
from s4 import features as ft

configurar_gpu()
cfg = cargar_config("config/ablacion_2025_2026.yaml")
datos = preparar_datos(cfg)
LOOKBACKS = [30, 60, 120, 180, 240, 360]
H = cfg["ventanas"]["horizon"]
com = {k: inicios_comunes(datos[k].index, LOOKBACKS, H) for k in ("train", "val", "test")}
for L in LOOKBACKS:
    c = variante(cfg, **{"ventanas.lookback": L})
    correr_variante(c, datos, ft.CONJUNTOS["B_S4_ciclica"], f"LB{L}", "lookback",
                    starts={k: com[k][L] for k in com})
```

**Decisión:** regla de parsimonia (la ventana más corta dentro de una desviación estándar de la mejor en HSS **y** en RMSE de eventos) → `ventanas.lookback: 60` en el YAML.

### 2.2 `scripts/correr_ablacion.py` — ablación principal (paso 4)

```python
import sys, warnings
sys.path.insert(0, "src"); warnings.filterwarnings("ignore")
from s4.config import cargar_config, variante, ruta
from s4.pipeline import preparar_datos, correr_variante, baselines, construir_particiones, evaluar_y_registrar
from s4.registry import leer, resumir_semillas
from s4.train import configurar_gpu
from s4 import features as ft

configurar_gpu()
cfg = cargar_config("config/ablacion_2025_2026.yaml")       # LOOKBACK = 60
datos = preparar_datos(cfg)

parts = construir_particiones(cfg, datos, ft.CONJUNTOS["A_S4"])
for nombre, yhat in baselines(cfg, datos, parts, "val").items():     # persistencia y climatología
    evaluar_y_registrar(cfg, parts["val"], yhat, nombre, "ablacion")

VARIANTES = [
    ("A · S4 + MSE",                 "A_S4",         {"perdida.tipo": "mse"}),
    ("B · + cíclica + MSE",          "B_S4_ciclica", {"perdida.tipo": "mse"}),
    ("C · + cíclica + WFL",          "B_S4_ciclica", {}),
    ("D · + TEC/ROTEC/ROTI + WFL",   "D_locales",    {}),
    ("D' · + solo ROTI + WFL",       "D_solo_ROTI",  {}),
    ("E · + índices globales + WFL", "E_completo",   {}),
]
for nombre, conj, ov in VARIANTES:
    correr_variante(variante(cfg, **ov), datos, ft.CONJUNTOS[conj], nombre, "ablacion")
```

### 2.3 `scripts/reparar_registro.py` — reparación del registro (paso 5)

Algunas filas del registro tenían distinto número de columnas (evaluaciones sin ventanas de inicio y baselines sin columnas de entrenamiento), y pandas las leía desplazadas: los filtros por grupo devolvían tablas vacías. El script reconstruye cada fila según su número de campos y guarda un respaldo `.respaldo`. Además se corrigieron `evaluate.anticipacion_inicio` (siempre devuelve 3 claves) y `registry.registrar` (alinea cada fila con la cabecera).

```bash
python scripts/reparar_registro.py experiments/registro_ablacion_2025_2026.csv
```

### 2.4 `scripts/correr_ablacion_extra.py` — ablación complementaria (paso 6)

Cierra dos huecos de la ablación principal: (1) la codificación cíclica solo se había probado con MSE; (2) la ventana de 60 min se eligió con S4 solo, y los precursores físicos (subida de densidad electrónica) ocurren 1–3 h antes del evento.

```python
VARIANTES = [
    ("A' · S4 + WFL (sin cíclica)",           "A_S4",         {}),
    ("C · + cíclica + WFL · LB180",           "B_S4_ciclica", {"ventanas.lookback": 180}),
    ("D' · + solo ROTI + WFL · LB180",        "D_solo_ROTI",  {"ventanas.lookback": 180}),
    ("E · + índices globales + WFL · LB180",  "E_completo",   {"ventanas.lookback": 180}),
]
for nombre, conj, ov in VARIANTES:
    correr_variante(variante(cfg, **ov), datos, ft.CONJUNTOS[conj], nombre, "ablacion_extra")
```

### Conjuntos de entrada (`src/s4/features.py`)

| Conjunto | Variables |
|---|---|
| `A_S4` | S4 |
| `B_S4_ciclica` | S4 + daily_sin, daily_cos, day_of_year_sin, day_of_year_cos |
| `D_locales` | B + TEC, ROTEC, ROTI |
| `D_solo_ROTI` | B + ROTI |
| `E_completo` | D_locales + Kp, Dst, AE, F10.7 |

## 3. Reporte a partir del registro (no entrena)

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, ruta
from s4.registry import leer, resumir_semillas

cfg = cargar_config("../config/ablacion_2025_2026.yaml")
REG = ruta(cfg, "experimentos") / f"registro_{cfg['nombre']}.csv"
MET = ("RMSE_global", "RMSE_evento", "RMSE_calma", "POD", "Precision", "HSS", "POD_inicio")
print("LOOKBACK en el YAML:", cfg["ventanas"]["lookback"], "min | registro:", REG)
reg = pd.read_csv(REG)
print(reg.groupby(["grupo", "particion"]).size())

### 3.1 Ablación principal (ventana de 60 min)

In [ ]:
abl = resumir_semillas(leer(REG, grupo="ablacion", particion="val"), por=("modelo",), metricas=MET)
orden = ["Persistencia", "Climatología diaria", "A · S4 + MSE", "B · + cíclica + MSE", "C · + cíclica + WFL",
         "D · + TEC/ROTEC/ROTI + WFL", "D' · + solo ROTI + WFL", "E · + índices globales + WFL"]
abl = abl.set_index("modelo").reindex([o for o in orden if o in set(abl["modelo"])]).reset_index()
abl.to_csv(ruta(cfg, "reportes") / "tables/tabla_ablacion.csv", index=False)
abl.round(3)

### 3.2 Ablación complementaria (S4 + WFL sin hora; precursores con 180 min)

In [ ]:
ext = resumir_semillas(leer(REG, grupo="ablacion_extra", particion="val"), por=("modelo",), metricas=MET)
ext.to_csv(ruta(cfg, "reportes") / "tables/tabla_ablacion_extra.csv", index=False)
ext.round(3)

### 3.3 Figura: habilidad de alerta (HSS) por variante

In [ ]:
todas = pd.concat([abl, ext], ignore_index=True)
t = todas.dropna(subset=["HSS_std"]).sort_values("HSS_mean")
fig, ax = plt.subplots(figsize=(10, 6))
colores = ["#0099FF" if "LB180" in m else "#0000AF" for m in t["modelo"]]
ax.barh(t["modelo"], t["HSS_mean"], xerr=t["HSS_std"], color=colores, capsize=4)
pers = todas.loc[todas["modelo"] == "Persistencia", "HSS_mean"]
if len(pers):
    ax.axvline(pers.iloc[0], color="#C77C02", ls="--", lw=2, label=f"Persistencia (HSS {pers.iloc[0]:.2f})")
ax.set_xlabel("HSS en validación (media ± desviación entre 3 semillas)")
ax.set_title("Estudio de ablación · azul oscuro: 60 min · celeste: 180 min")
ax.legend(loc="lower right"); plt.tight_layout()
plt.savefig(ruta(cfg, "reportes") / "figures/fig_ablacion_hss.png", dpi=200); plt.show()

### 3.4 Figura: detección frente a precisión (el compromiso de la WFL)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
desv = [(8, 8), (8, -14), (-58, 8), (-58, -14), (8, 20), (-58, 20)]
for i, (_, r) in enumerate(todas.iterrows()):
    if pd.isna(r["POD_mean"]) or pd.isna(r["Precision_mean"]):
        continue
    ax.scatter(r["POD_mean"], r["Precision_mean"], s=120,
               color="#C77C02" if r["modelo"] == "Persistencia" else ("#0099FF" if "LB180" in r["modelo"] else "#0000AF"))
    ax.annotate(r["modelo"].split(" · ")[0] + (" (180)" if "LB180" in r["modelo"] else ""),
                (r["POD_mean"], r["Precision_mean"]), textcoords="offset points", xytext=desv[i % len(desv)], fontsize=9,
                arrowprops=dict(arrowstyle="-", color="#999999", lw=0.6))
ax.set_xlabel("POD: ventanas con evento detectadas"); ax.set_ylabel("Precisión de la alerta")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05); ax.grid(alpha=0.3)
ax.set_title("Con MSE el modelo casi no alerta; la WFL desplaza el punto hacia la detección")
plt.tight_layout(); plt.savefig(ruta(cfg, "reportes") / "figures/fig_ablacion_precision_pod.png", dpi=200); plt.show()

## 4. Resultados obtenidos (validación, 3 semillas)

### 4.1 Ablación principal (60 min)

| Variante | HSS | POD | Precisión | Inicios anticipados | RMSE eventos | RMSE global |
|---|---|---|---|---|---|---|
| Persistencia | 0.333 | 21 % | 90 % | 0 % | 0.286 | 0.049 |
| A · S4 + MSE | 0.178 ± 0.022 | 10 % | 97 % | 0.7 % | 0.341 | 0.042 |
| B · + cíclica + MSE | 0.151 ± 0.033 | 8 % | 98 % | 0.6 % | 0.348 | 0.041 |
| **C · + cíclica + WFL** | **0.556 ± 0.012** | 81 % | 43 % | 77 % | **0.148** | 0.083 |
| D · + TEC/ROTEC/ROTI | 0.551 ± 0.059 | 84 % | 42 % | 80 % | 0.153 | 0.087 |
| D' · + solo ROTI | 0.555 ± 0.031 | 82 % | 43 % | 77 % | 0.148 | 0.084 |
| E · + índices globales | 0.561 ± 0.018 | 80 % | 44 % | 75 % | 0.153 | 0.088 |

### 4.2 Ablación complementaria

| Variante | HSS | POD | Precisión | Inicios anticipados |
|---|---|---|---|---|
| A' · S4 + WFL, sin hora (60 min) | 0.525 ± 0.010 | 89 % | 38 % | 87 % |
| C · + hora (180 min) | 0.547 ± 0.018 | 81 % | 42 % | 77 % |
| **D' · + ROTI (180 min)** | **0.589 ± 0.019** | 81 % | 47 % | 76 % |
| E · + todos los precursores (180 min) | 0.595 ± 0.032 | 75 % | 51 % | 69 % |

## 5. Conclusiones

1. **La Weighted Focal Loss es el componente decisivo.** De B a C (mismas entradas, misma red) el HSS pasa de 0.15 a 0.56, la detección de 8 % a 81 % y la anticipación de inicios de 0.6 % a 77 %. Con MSE el modelo predice "calma" y queda por debajo de la persistencia. Costo: el RMSE global se duplica.
2. **La codificación cíclica aporta solo con la WFL.** Con MSE no hay efecto medible (A → B); con WFL (A' → C) sube el HSS de 0.52 a 0.56 y la precisión de 38 % a 43 %: la hora permite descartar alertas en horas sin formación de burbujas.
3. **Los precursores no aportan.** Con 60 min no mejoran al modelo. Con 180 min el HSS pareció subir de 0.55 a 0.59 al agregar ROTI, pero la verificación posterior lo descartó: en el notebook 08, al desordenar ROTI el HSS no disminuye (−0.007 ± 0.004) y ROTI explica solo el 2.5 % de las decisiones; y con la configuración final (α = 1.5, β = 10, umbral calibrado), S4 + hora obtiene 0.658 ± 0.003 (60 min) frente a 0.661 ± 0.009 con ROTI. La mejora aparente era variabilidad entre corridas (≈ 0.03 por el no-determinismo de la GPU).
4. **La anticipación viene del S4 reciente y de la hora del día:** al desordenar la hora se pierden 26 puntos de inicios anticipados (notebook 08). La red aprendió la climatología del centelleo post-crepuscular.
5. **Modelo final:** C · S4 + hora · ventana de 60 min · α = 1.5, β = 10 · umbral 0.55 (notebooks 06 y 08), a evaluar una sola vez en Test (notebook 09).

**Limitaciones.** Resultados de validación con 3 semillas; las diferencias de los precursores (≈ +0.04 de HSS) equivalen a unas 2–3 desviaciones estándar y deben confirmarse en Test. Las variantes de 60 y 180 min usan conjuntos de ventanas ligeramente distintos (más historia exige más continuidad de datos).